In [1]:
import pandas as pd

df = pd.read_csv("../data/ab_user_level_final.csv")

In [2]:
# TODO:
# impression, click, purchase, revenue 컬럼의 제곱값을 각각 생성하세요.
# 이후 그룹별 제곱합(sum of squares)을 계산하기 위해 필요합니다.
df["impression_sq"] = df["impression"] ** 2
df["click_sq"] = df["click"] ** 2
df["purchase_sq"] = df["purchase"] ** 2
df["revenue_sq"] = df["revenue"] ** 2

In [3]:
# variant, country, device 기준으로 그룹화하여
# user 수, 각 지표의 합계, 각 지표의 제곱합을 집계합니다.
agg_df = df.groupby(["variant", "country", "device"]).agg({
    "user_id": "count",
    "impression": "sum",
    "click": "sum",
    "purchase": "sum",
    "revenue": "sum",
    "impression_sq": "sum",
    "click_sq": "sum",
    "purchase_sq": "sum",
    "revenue_sq": "sum"
}).reset_index()



In [4]:
# TODO:
# user_id 집계 컬럼 이름을 count_n으로 변경하세요.
agg_df = agg_df.rename(columns={'user_id':'count_n'})

cube_parts = []

metrics = [
    ("impression", "impression", "impression_sq"),
    ("click", "click", "click_sq"),
    ("purchase", "purchase", "purchase_sq"),
    ("revenue", "revenue", "revenue_sq")
]

for metric_name, value_col, sq_col in metrics:
    temp = agg_df[[
        "variant", "country", "device", "count_n",
        value_col, sq_col
    ]].copy()

    temp["metric"] = metric_name

    # TODO:
    # value_col은 sum_value로, sq_col은 sum_of_squares로 이름을 변경하세요.
    temp = temp.rename(columns={value_col:"sum_value", sq_col: "sum_of_squares"})

    temp = temp[[
        "variant", "country", "device",
        "metric", "count_n", "sum_value", "sum_of_squares"
    ]]

    cube_parts.append(temp)

In [5]:
# TODO:
# cube_parts를 하나의 데이터프레임으로 합쳐 cube_df를 생성하세요.
cube_df = pd.concat(cube_parts, ignore_index=True)

print(cube_df.shape)
print(cube_df.head(10))
print(cube_df["metric"].value_counts())

# TODO:
# 최종 cube_df를 CSV 파일로 저장하세요.
cube_df.to_csv("cube_analysis_result.csv", index=False)

(96, 7)
   variant country   device      metric  count_n  sum_value  sum_of_squares
0        0      DE  desktop  impression       81      125.0           229.0
1        0      DE   mobile  impression      128      212.0           414.0
2        0      DE   tablet  impression       20       33.0            65.0
3        0      ES  desktop  impression       36       64.0           136.0
4        0      ES   mobile  impression       94      158.0           312.0
5        0      ES   tablet  impression        6       11.0            23.0
6        0      FR  desktop  impression       55       91.0           173.0
7        0      FR   mobile  impression      130      215.0           415.0
8        0      FR   tablet  impression       14       25.0            51.0
9        0      UK  desktop  impression      128      208.0           398.0
metric
impression    24
click         24
purchase      24
revenue       24
Name: count, dtype: int64
